# 🗺️ 33 · Spatial omics: tissues as data
### Spatially variable genes, tissue domains, cell neighbourhoods, niche effects and the limits of ligand–receptor analysis, deconvolution of mixed spots and segmentation errors — on a simulated tumour section with a known answer key

**BiologyToML** · Level 2 · Research frontier · Part G — Cells and tissues · Module 33 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/33_spatial_omics.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (sparse matrices, k-d trees, non-negative least squares), pandas, Matplotlib, scikit-learn (nearest neighbours, k-means, ARI) |
| **Prerequisites** | Level 1 (especially Modules 06–07, 12, 14, 16 and 18) and [Module 30](30_single_cell_rnaseq.ipynb) (normalisation, PCA, clustering of single cells); [Module 25](25_graph_neural_networks.ipynb) helps for Section 3 but is not required |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [06 · Linear regression](06_linear_regression_gwas.ipynb), [07 · Ridge, lasso and constraints](07_ridge_lasso_sparse_regression.ipynb), [12 · Clustering](12_clustering_kmeans_hierarchical.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [16 · Conditional independence and permutation tests](16_bayesian_networks_independence.ipynb), [18 · Gaussian graphical models](18_gaussian_graphical_models.ipynb), [25 · Graph neural networks](25_graph_neural_networks.ipynb), [30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb) |
| **Research frontier covered** | spatial transcriptomics (imaging- and sequencing-based), spatially variable genes, spatial domains, cell–cell neighbourhoods and communication, spot deconvolution — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED** with the toolkit's `simulate_spatial_tissue`: a square section of a tumour with 4,000
> cells measured at single-cell resolution, and the same cells pooled into Visium-like spots. The hidden truth includes
> each cell's type and tissue domain (tumour core, invasive margin, normal epithelium, stroma), a hypoxia programme that
> rises towards the tumour centre, an exhaustion programme in T cells that grows with the number of nearby tumour cells,
> and the true cell-type composition of every spot. Gene names follow real biology so that plots read naturally; the
> values are simulated. Real sections add tissue folds and tears, uneven capture, diffusion of transcripts, imperfect
> segmentation and no answer key — see Section 8.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Compare** imaging-based and sequencing-based spatial transcriptomics — resolution, gene coverage, typical artefacts — and **explain** what each measures.
2. **Compute** Moran's I with spatial weights, **test** it by permutation, and **show** why spatially variable genes are not the same as highly variable genes.
3. **Find** tissue domains by clustering expression smoothed over spatial neighbours (a graph convolution) and **measure** the gain over expression-only clustering with the ARI.
4. **Quantify** cell-type co-localisation with permutation z-scores and **check** them against the simulated design.
5. **Detect** a niche effect (T-cell exhaustion near tumour cells) with regression and conditional permutation tests, and **demonstrate** why co-localisation is not evidence of ligand–receptor signalling.
6. **Deconvolve** mixed spots with non-negative least squares and a Poisson model fitted by multiplicative updates, and **diagnose** the bias caused by a cell type missing from the reference.
7. **Simulate** segmentation errors and **show** how they cause mis-annotation and spurious niche effects.
8. **Connect** each step to Squidpy, Giotto, SpaGCN, BayesSpace, RCTD and cell2location.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Technologies and the SIMULATED section | imaging vs sequencing; cells, domains, spots, gene maps | 15 min |
| 2 · Spatial statistics | Moran's I, permutation tests; 🧪 SVGs vs HVGs | 25 min |
| 3 · Spatial domains | neighbour averaging and graph convolution; 🧪 ARI | 20 min |
| 4 · Neighbourhoods | co-localisation z-scores; 🧪 vs the design | 15 min |
| 5 · Communication and niche effects | exhaustion vs tumour neighbours; ligand–receptor caveats | 30 min |
| 6 · Deconvolution of spots | NNLS, Poisson multiplicative updates; 🧪 RMSE; missing types | 30 min |
| 7 · Segmentation and resolution | spillover, mis-annotation, spurious niches | 15 min |
| 8 · 🧭 From toy to real data | Squidpy, Giotto, cell2location, RCTD; public data | 10 min |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint, discussion, going further, lab note | 15 min |

## 🧫 The biological question

A tumour is not a bag of cancer cells. It is a tissue: cancer cells packed into a core where oxygen runs out, an
**invasive margin** where they meet the surrounding tissue, blood vessels, fibroblasts that lay down matrix, and immune
cells — T cells that may kill tumour cells, or that may become **exhausted** after prolonged contact with them, and
macrophages that can help either side. Where an immune cell sits matters: in several cancer types, tumours whose T
cells infiltrate the tumour itself tend to respond better to immunotherapy than tumours whose T cells are held back in
the surrounding stroma. None of this is
visible after dissociation: single-cell RNA-seq ([Module 30](30_single_cell_rnaseq.ipynb)) tells us *which* cells are
present, not *where* they were.

**Spatial transcriptomics** measures gene expression in place, on a tissue section. It was named *Method of the Year
2020* by *Nature Methods* (Marx 2021), and atlases of tumours, brains and developing organs now routinely include
spatial data. The questions are new kinds of statistics:

1. Which genes have a **spatial pattern**, and which only vary from cell to cell?
2. Can we find the tissue's **domains** (core, margin, stroma) from the data?
3. Which cell types **live together**, and does a cell's **neighbourhood** change its state?
4. When each measurement spot contains several cells, can we **unmix** it?

Our SIMULATED section mimics a carcinoma with a tumour core, an immune-rich invasive margin, a band of normal epithelium
and stroma — the layout studied in many spatial tumour atlases. Two spatial programmes are planted: hypoxia genes rise
towards the tumour centre, and T cells become exhausted in proportion to the number of tumour cells around them.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle
from scipy import sparse, stats
from scipy.optimize import linear_sum_assignment, nnls
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_spatial_tissue` generates the tumour section with its hidden truth; `TISSUE_TYPES`, `HYPOXIA_GENES` and `EXHAUSTION_GENES` name the cell types and the two spatial programmes; `log_normalize`, `pca` and `benjamini_hochberg` are the preprocessing and testing helpers of [Module 30](30_single_cell_rnaseq.ipynb); `plot_embedding` draws cells coloured by a label. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


TISSUE_TYPES = {
    "Tumor": ("epithelial", ["EPCAM", "KRT8", "KRT18", "CEACAM5"], 2.4),
    "Epithelial": ("epithelial", ["KRT5", "KRT14", "TP63", "KRT15"], 2.2),
    "Fibroblast": ("stromal", ["COL1A1", "COL1A2", "DCN", "LUM"], 2.4),
    "Endothelial": ("stromal", ["PECAM1", "VWF", "CLDN5", "CDH5"], 2.2),
    "T cell": ("immune", ["CD3E", "CD8A", "GZMB", "NKG7"], 2.0),
    "Macrophage": ("immune", ["CD68", "CD163", "C1QA", "C1QB"], 2.2),
}


HYPOXIA_GENES = ["VEGFA", "CA9", "SLC2A1", "LDHA", "BNIP3", "PGK1"]


EXHAUSTION_GENES = ["PDCD1", "HAVCR2", "LAG3", "TOX"]


def simulate_spatial_tissue(rng: np.random.Generator, n_cells: int = 4000, n_genes: int = 300,
                            spot_spacing: float = 0.05, depth: float = 400) -> dict:
    """Simulate a spatial-transcriptomics section of a tumour (unit square, single-cell resolution).

    Domains: *tumour core* (a disc), *invasive margin* (a ring around it), *normal epithelium*
    (a band along the top edge) and *stroma* (everything else); each domain has its own
    cell-type composition.  Spatial signals beyond cell type: a hypoxia programme that rises
    towards the tumour centre, and an exhaustion programme in T cells that grows with the
    number of tumour cells nearby (cell–cell communication).  Cells are also aggregated
    into Visium-like spots (hexagonal grid, spacing ``spot_spacing``).

    The programme genes have a fixed baseline in every cell type (hypoxia genes moderate,
    exhaustion genes low), so their spatial patterns come only from the planted programmes.

    Returns a dict with ``cells`` (x, y, domain, cell_type), ``counts`` (cells × genes),
    ``var``, ``spots`` (x, y, number of cells), ``spot_counts``, ``spot_proportions``
    (true cell-type fractions per spot) and ``reference`` (mean expression profile per type).
    """
    names = list(TISSUE_TYPES)
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, TISSUE_TYPES, n_extra_markers=12,
                                                          extra_names=HYPOXIA_GENES + EXHAUSTION_GENES)
    profiles[:, np.isin(genes, HYPOXIA_GENES)] = 0.5                     # moderate baseline everywhere
    profiles[:, np.isin(genes, EXHAUSTION_GENES)] = -1.0                 # low baseline everywhere
    x, y = rng.random(n_cells), rng.random(n_cells)
    centre, radius = np.array([0.62, 0.42]), 0.2
    r = np.hypot(x - centre[0], y - centre[1])
    domain = np.where(r < radius, "tumor core", np.where(r < radius + 0.08, "invasive margin",
                      np.where(y > 0.85 + 0.03 * np.sin(8 * x), "epithelium", "stroma")))
    composition = {  # Tumor, Epithelial, Fibroblast, Endothelial, T cell, Macrophage
        "tumor core": [0.80, 0.00, 0.03, 0.07, 0.03, 0.07],
        "invasive margin": [0.25, 0.00, 0.15, 0.05, 0.30, 0.25],
        "epithelium": [0.00, 0.85, 0.10, 0.02, 0.02, 0.01],
        "stroma": [0.00, 0.02, 0.62, 0.16, 0.08, 0.12],
    }
    cell_type = np.array([rng.choice(len(names), p=composition[d]) for d in domain])
    log_rate = profiles[cell_type].copy()
    hyp = np.isin(genes, HYPOXIA_GENES)
    log_rate[:, hyp] += 3.0 * np.clip(1 - r / radius, 0, 1)[:, None] * (cell_type == 0)[:, None]
    from scipy.spatial import cKDTree
    tree = cKDTree(np.c_[x, y])
    n_tumor_nb = np.array([np.sum(cell_type[nb] == 0) for nb in tree.query_ball_point(np.c_[x, y], 0.04)])
    exh = np.isin(genes, EXHAUSTION_GENES)
    log_rate[:, exh] += (0.5 * np.minimum(n_tumor_nb, 8) * (cell_type == names.index("T cell")))[:, None]
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    lib = depth * np.exp(rng.normal(0, 0.3, n_cells))
    counts = sample_negative_binomial(rng, lib[:, None] * prob, 10.0)
    # Visium-like spots on a hexagonal grid
    sx, sy = np.meshgrid(np.arange(spot_spacing / 2, 1, spot_spacing), np.arange(spot_spacing / 2, 1, spot_spacing * 0.866))
    sx = sx + (np.arange(sx.shape[0]) % 2)[:, None] * spot_spacing / 2
    spot_xy = np.c_[sx.ravel(), sy.ravel()]
    spot_xy = spot_xy[(spot_xy[:, 0] < 1) & (spot_xy[:, 1] < 1)]
    members = tree.query_ball_point(spot_xy, spot_spacing * 0.45)
    spot_counts = np.array([counts[m].sum(axis=0) if m else np.zeros(len(genes), int) for m in members])
    props = np.array([np.bincount(cell_type[m], minlength=len(names)) / max(len(m), 1) for m in members])
    cells = pd.DataFrame({"x": x, "y": y, "domain": domain, "cell_type": np.array(names, dtype=object)[cell_type],
                          "n_tumor_neighbours": n_tumor_nb}, index=[f"cell{i:05d}" for i in range(n_cells)])
    spots = pd.DataFrame({"x": spot_xy[:, 0], "y": spot_xy[:, 1], "n_cells": [len(m) for m in members]},
                         index=[f"spot{i:04d}" for i in range(len(spot_xy))])
    reference = pd.DataFrame(np.exp(profiles) / np.exp(profiles).sum(axis=1, keepdims=True), index=names, columns=genes)
    var = pd.DataFrame({"marker_of": marker_of, "is_hypoxia": hyp, "is_exhaustion": exh}, index=genes)
    return dict(cells=cells, counts=counts, var=var, spots=spots, spot_counts=spot_counts.astype(np.int32),
                spot_proportions=pd.DataFrame(props, index=spots.index, columns=names), reference=reference)


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def benjamini_hochberg(pvalues) -> np.ndarray:
    """Benjamini–Hochberg adjusted p-values (q-values): reject where q ≤ α to control the FDR at α."""
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(q, 1.0)
    return out


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_spatial_tissue', 'log_normalize', 'pca', 'benjamini_hochberg', 'plot_embedding']))

---
## 1 · Technologies and the SIMULATED section

> 📍 **Where we are.** Everything in Modules 30–32 started from a dissociated cell suspension. Spatial methods keep
> the tissue intact and add two numbers to every measurement: its $x$ and $y$ coordinates. First, how is that done?

### 🔬 Assay box — two families of spatial transcriptomics

| | **Imaging-based** (MERFISH, seqFISH, Xenium, CosMx, MERSCOPE) | **Sequencing-based** (Spatial Transcriptomics, Visium, Slide-seq, Stereo-seq, Visium HD) |
|---|---|---|
| **How** | Fluorescent probes hybridise to chosen mRNAs in the section; repeated rounds of imaging read out a barcode for each molecule, which is located under the microscope | The section is placed on a surface carrying spatially barcoded capture oligos (spots, beads or a dense array); mRNAs diffuse onto it, are tagged with the barcode of their position and are sequenced |
| **Genes** | A targeted panel: hundreds to a few thousand genes chosen in advance | Whole transcriptome (anything polyadenylated) |
| **Resolution** | Single molecules; cells must be **segmented** from images to get per-cell counts | Set by the spot size: Visium spots are 55 µm across and usually contain several cells; Slide-seq beads are 10 µm; Visium HD and Stereo-seq reach (sub)cellular bins |
| **Typical artefacts** | Segmentation errors (transcripts assigned to the wrong cell), optical crowding, panel bias, autofluorescence | **Mixtures** of cell types in each spot, lateral diffusion of mRNA, low capture per spot, tissue folds and edge effects |

The two families are complementary: imaging gives cells but few genes; sequencing gives all genes but mixtures (or, at
the finest resolutions, very sparse counts). Our simulator provides *both views of the same section*: single-cell counts
(as from imaging) and Visium-like spot counts obtained by adding up the cells under each spot.

In [ ]:
# ── 1.1 The SIMULATED tumour section ──
tissue = simulate_spatial_tissue(np.random.default_rng(SEED + 1))
cells_df, counts, var = tissue["cells"], tissue["counts"], tissue["var"]
xy = cells_df[["x", "y"]].to_numpy()
ctype, domain = cells_df.cell_type.to_numpy(str), cells_df.domain.to_numpy(str)
genes = var.index.to_numpy(str)
TYPES = list(TISSUE_TYPES)
gidx = {g: j for j, g in enumerate(genes)}
print(f"SIMULATED section: {counts.shape[0]} cells × {counts.shape[1]} genes at single-cell resolution")
print(f"UMIs per cell: median {np.median(counts.sum(1)):.0f}; zeros: {np.mean(counts == 0):.0%} of entries")
print("\ncells per domain and cell type (hidden truth):\n" + pd.crosstab(cells_df.domain, cells_df.cell_type).to_string())
spots, spot_counts = tissue["spots"], tissue["spot_counts"]
print(f"\nVisium-like spots: {len(spots)} (spacing 0.05, cells per spot: median {spots.n_cells.median():.0f}, "
      f"range {spots.n_cells.min()}–{spots.n_cells.max()}); spot × gene matrix {spot_counts.shape}")
print("genes of the planted programmes:", HYPOXIA_GENES, EXHAUSTION_GENES)

In [ ]:
# ── 1.2 Maps: cell types, domains, and the same tissue seen through spots ──
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
plot_embedding(xy, ctype, ax=axes[0], s=3, title="cell types (hidden truth)")
plot_embedding(xy, domain, ax=axes[1], s=3, title="tissue domains (hidden truth)")
for ax in axes[:2]:                                         # legends below the maps, not on top of the cells
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=7, markerscale=3, frameon=False)
win = (xy[:, 0] > 0.3) & (xy[:, 0] < 0.55) & (xy[:, 1] > 0.3) & (xy[:, 1] < 0.55)   # zoom on the invasive margin
palette = plt.get_cmap("tab10")
for k, t in enumerate(sorted(set(ctype))):
    m = win & (ctype == t)
    axes[2].scatter(*xy[m].T, s=14, color=palette(k), label=t)
sw = (spots.x > 0.3) & (spots.x < 0.55) & (spots.y > 0.3) & (spots.y < 0.55)
for sx, sy in spots.loc[sw, ["x", "y"]].to_numpy():
    axes[2].add_patch(Circle((sx, sy), 0.05 * 0.45, fill=False, ec="k", lw=0.8))
axes[2].set_xlim(0.3, 0.55); axes[2].set_ylim(0.3, 0.55); axes[2].set_xticks([]); axes[2].set_yticks([])
axes[2].set_title("zoom on the margin: cells and spots (circles)")
for ax in axes:
    ax.set_aspect("equal")
plt.suptitle("SIMULATED tumour section (unit square)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The section has a disc-shaped **tumour core** (mostly tumour cells), a thin **invasive
margin** ring around it where tumour cells, T cells and macrophages mix, a band of **normal epithelium** along the top
edge and **stroma** (fibroblasts, endothelial cells and scattered immune cells) everywhere else. Each cell has about 400
UMIs and three quarters of the entries are zero — shallower than the droplet data of Module 30 (about 2,000 UMIs per
cell); single-cell spatial data are often shallow per cell. The zoom shows the sequencing view: each circle is one spot that adds up the transcripts of the cells under
it — here a median of 6 cells, of several types. A spot at the margin can contain tumour cells, T cells and macrophages
at once.

In [ ]:
# ── 1.3 Gene expression in place ──
L = log_normalize(counts)                                   # log(1 + 10^4 × count / total), as in Module 30
show = {"EPCAM": "tumour", "COL1A1": "fibroblasts", "PGK1": "hypoxia", "PDCD1": "exhaustion; T cells only"}
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, (g, what) in zip(axes, show.items()):
    m = (ctype == "T cell") if g == "PDCD1" else np.ones(len(xy), bool)
    ax.scatter(*xy[~m].T, s=2, color="0.9")                 # cells not shown in colour
    order = np.flatnonzero(m)[np.argsort(L[m, gidx[g]])]    # draw high values on top
    sc = ax.scatter(*xy[order].T, c=L[order, gidx[g]], s=2 if g != "PDCD1" else 6, cmap="magma_r", vmin=0, vmax=9)
    ax.set_title(f"{g} ({what})"); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, shrink=0.8, label="log-normalised expression")
plt.suptitle("SIMULATED expression maps", y=1.02)
plt.show()

🔍 **What just happened?** Each map is one gene, one dot per cell. The tumour marker outlines the tumour core and the
tumour cells of the margin; the fibroblast marker fills the stroma. The hypoxia gene is expressed at a moderate level
everywhere and rises in the inner part of the core — the simulated oxygen gradient. The exhaustion gene `PDCD1`, shown
in T cells only (other cells barely express it), is switched on in the T cells inside and at the edge of the tumour,
while T cells in the stroma stay silent — the planted niche effect. Our eyes find these patterns at once; the next sections turn them into numbers and tests.

✏️ **Your turn.** Map one of the other hypoxia genes (`VEGFA` or `CA9`) and one of the other exhaustion genes (`HAVCR2`,
`LAG3`). Can you see the patterns? Print the mean count of each gene per cell — what does that predict for Section 2?

---
## 2 · Spatial statistics: which genes have a spatial pattern?

> 📍 **Where we are.** In [Module 30](30_single_cell_rnaseq.ipynb) we selected *highly variable* genes — genes that
> differ between cells. A spatially variable gene is different: its value at one place predicts its value nearby.

**Spatial weights.** We connect each cell to its $k = 6$ nearest neighbours in the tissue and write the connections in a
matrix $W$ with $w_{ij} = 1/k$ if $j$ is one of $i$'s neighbours and 0 otherwise (each row sums to 1, so $(Wx)_i$ is the
average of $x$ over $i$'s neighbours).

🧮 **Math refresher — Moran's I.** For one gene with values $x_1, \dots, x_n$ and deviations $z_i = x_i - \bar x$,

$$I = \frac{n}{\sum_{ij} w_{ij}}\;\frac{\sum_{ij} w_{ij}\, z_i z_j}{\sum_i z_i^2} \;=\; \frac{\sum_i z_i\,(Wz)_i}{\sum_i z_i^2}\quad\text{(rows of } W \text{ sum to 1)} .$$

*In words:* Moran's I (Moran 1950) is a correlation between each cell's value and the average value of its neighbours.
$I > 0$: neighbours are alike (patches, gradients); $I \approx 0$: no spatial pattern (its expected value under random
placement is $-1/(n-1)$, essentially 0); $I < 0$: neighbours tend to differ (a checkerboard). It is one number per gene,
cheap to compute for thousands of genes.

**Testing by permutation.** Is an observed $I = 0.03$ meaningful? If position did not matter, any reassignment of the
expression values to positions would be equally likely. Shuffling the values over the cells many times and recomputing
$I$ gives the **null distribution**; the observed value is compared with it.

> 🔗 **Builds on** [Module 16](16_bayesian_networks_independence.ipynb), which tested conditional-independence
> predictions with a permutation test, and [Module 18](18_gaussian_graphical_models.ipynb): a gene with spatial
> autocorrelation is a field in which each cell depends on its neighbours — a **Markov random field** on the tissue
> graph, the undirected models of Module 18 laid out in space.

In [ ]:
# ── 2.1 Moran's I, first on a toy: four cells in a row ──
def morans_i(X, W):
    """Moran's I of every column of X (cells × genes) for a row-standardised sparse weight matrix W."""
    Zc = X - X.mean(axis=0)
    return np.asarray((Zc * (W @ Zc)).sum(axis=0) / (Zc ** 2).sum(axis=0)).ravel()


W_toy = sparse.csr_matrix(np.array([[0, 1, 0, 0], [0.5, 0, 0.5, 0], [0, 0.5, 0, 0.5], [0, 0, 1, 0]]))  # a path 1–2–3–4
for name, x in [("patch  ", [1, 1, 0, 0]), ("checker", [1, 0, 1, 0]), ("single ", [1, 0, 0, 0])]:
    print(f"toy {name} {x}: I = {morans_i(np.array(x, float)[:, None], W_toy)[0]:+.3f}")

K_SP = 6
nbr6 = NearestNeighbors(n_neighbors=K_SP + 1).fit(xy).kneighbors(xy)[1][:, 1:]
n_cells = len(xy)
W = sparse.csr_matrix((np.full(n_cells * K_SP, 1 / K_SP), (np.repeat(np.arange(n_cells), K_SP), nbr6.ravel())),
                      shape=(n_cells, n_cells))
print(f"\nspatial weights: {n_cells} cells, {K_SP} neighbours each (rows sum to {W.sum(axis=1).max():.0f})")

🔍 **What just happened?** On a path of four cells, a patch of high values on one side gives $I = +0.5$ (neighbours
alike), a checkerboard gives $I = -1$ (neighbours always different), and a single high cell gives a negative value too —
it differs from its only neighbour. On the tissue, the weight matrix links each of the 4,000 cells to its 6 nearest
neighbours.

### A trap for highly-variable-gene selection

To see the difference between *variable* and *spatially variable*, we add eight SIMULATED **decoy genes** that mimic a
cell-cycle programme: strongly expressed in a random 15 % of cells, wherever they are. They vary a lot from cell to cell
but have no spatial pattern by construction.

In [ ]:
# ── 2.2 Add decoys, compute Moran's I for every gene and test it by permutation ──
rng2 = np.random.default_rng(SEED + 2)
cycling = rng2.random(n_cells) < 0.15                       # random cells, anywhere in the tissue
lib = counts.sum(axis=1)
decoys = rng2.poisson(lib[:, None] * np.where(cycling[:, None], 0.015, 0.0003) * np.ones((1, 8)))
L_all = log_normalize(np.c_[counts, decoys])
names_all = np.r_[genes, [f"CYC{i + 1}" for i in range(8)]]

t0 = time.time()
I_obs = morans_i(L_all, W)
rng_perm = np.random.default_rng(SEED + 3)
null = np.array([morans_i(L_all[rng_perm.permutation(n_cells)], W) for _ in range(199)])   # 199 shuffles
z_I = (I_obs - null.mean(axis=0)) / null.std(axis=0)
q_I = benjamini_hochberg(stats.norm.sf(z_I))                # one-sided: we look for positive autocorrelation
print(f"Moran's I for {len(names_all)} genes with 199 permutations in {time.time() - t0:.1f} s")
print(f"null distribution: mean {null.mean():+.4f} (theory −1/(n−1) = {-1 / (n_cells - 1):+.4f}), "
      f"typical sd {null.std(axis=0).mean():.4f}")
print(f"spatially variable genes at FDR 5 %: {np.sum(q_I < 0.05)}")

In [ ]:
# ── 2.2b What a permutation test looks like, for one patterned gene and one decoy ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, g in zip(axes, ["PGK1", "CYC1"]):
    j = int(np.flatnonzero(names_all == g)[0])
    ax.hist(null[:, j], bins=25, color="0.7", label="199 shuffles (null)")
    ax.axvline(I_obs[j], color=CLASS_COLORS[1], lw=2, label=f"observed I = {I_obs[j]:.3f}")
    ax.set_xlabel("Moran's I"); ax.set_title(f"{g}: z = {z_I[j]:.1f}, q = {q_I[j]:.1e}"); ax.legend(fontsize=7)
axes[0].set_ylabel("number of shuffles")
plt.suptitle("SIMULATED: observed Moran's I against its permutation null", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the hypoxia gene `PGK1` the observed $I$ lies far to the right of every one of the 199
shuffles: its spatial pattern cannot be produced by chance placement. For the decoy, the observed value lies well
inside its null distribution (z ≈ 0.4) — as it should for a gene whose expression ignores position. Notice how narrow the
null is (a standard deviation of about 0.009): with 4,000 cells even small autocorrelations stand out.

In [ ]:
# ── 2.3 🧪 Which genes were found? Spatially variable vs highly variable ──
ref = tissue["reference"].to_numpy()                        # true mean profile of each cell type (hidden truth)
fold_range = ref.max(axis=0) / ref.min(axis=0)
marker = var.marker_of.to_numpy(str) != ""
category = np.full(len(names_all), "other (differs between types)", dtype=object)
category[:len(genes)][(fold_range < 2) & ~marker] = "< 2-fold between cell types"
category[:len(genes)][marker] = "cell-type marker"
category[np.isin(names_all, HYPOXIA_GENES)] = "hypoxia programme"
category[np.isin(names_all, EXHAUSTION_GENES)] = "exhaustion programme"
category[len(genes):] = "decoy (variable, random in space)"
tab = pd.DataFrame({"category": category, "Moran's I": I_obs, "significant": q_I < 0.05, "variance": L_all.var(axis=0)})
print("🧪 detection by (hidden) gene category:")
summary = tab.groupby("category").agg(genes=("significant", "size"), median_I=("Moran's I", "median"),
                                      fraction_significant=("significant", "mean"), median_variance=("variance", "median"))
print(summary.round(3).to_string())
for crit in ["Moran's I", "variance"]:
    print(f"\ntop 30 genes by {crit}:", tab.sort_values(crit, ascending=False).head(30).category.value_counts().to_dict())
low = tab[tab.category.isin(["hypoxia programme", "exhaustion programme"])].copy()
low["mean count per cell"] = np.c_[counts, decoys].mean(axis=0)[low.index]
print("\nthe planted programmes gene by gene:\n" + low.set_index(names_all[low.index]).round(3).to_string())
prog_scores = np.c_[L[:, np.isin(genes, HYPOXIA_GENES)].mean(axis=1), L[:, np.isin(genes, EXHAUSTION_GENES)].mean(axis=1)]
I_sc = morans_i(prog_scores, W)
null_sc = np.array([morans_i(prog_scores[rng_perm.permutation(n_cells)], W) for _ in range(199)])
z_sc = (I_sc - null_sc.mean(axis=0)) / null_sc.std(axis=0)
print(f"programme scores (mean of each programme's genes): hypoxia I = {I_sc[0]:.3f} (z = {z_sc[0]:.1f}), "
      f"exhaustion I = {I_sc[1]:.3f} (z = {z_sc[1]:.1f})")

In [ ]:
# ── 2.4 Variance vs Moran's I ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
cats = ["< 2-fold between cell types", "other (differs between types)", "cell-type marker", "hypoxia programme",
        "exhaustion programme", "decoy (variable, random in space)"]
for k, c in enumerate(cats):
    m = category == c
    axes[0].scatter(L_all.var(axis=0)[m], I_obs[m], s=14 if k > 2 else 7, color=CLASS_COLORS[k], label=c,
                    marker="x" if c.startswith("decoy") else "o")
axes[0].axhline(np.quantile(null, 0.99), color="k", ls="--", lw=1, label="99th percentile of the permutation null")
axes[0].set_xlabel("variance of log-normalised expression (HVG criterion)"); axes[0].set_ylabel("Moran's I")
axes[0].set_title("variable ≠ spatially variable"); axes[0].legend(fontsize=6.5)
d = len(genes)                                              # first decoy
sc = axes[1].scatter(*xy.T, c=L_all[:, d], s=2, cmap="magma_r")
axes[1].set_aspect("equal"); axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_title(f"decoy CYC1: variance {L_all[:, d].var():.2f}, Moran's I {I_obs[d]:+.3f}")
plt.suptitle("SIMULATED: spatially variable genes vs highly variable genes", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The permutation null behaves as theory says (mean ≈ −1/(n−1) ≈ 0). With 199 shuffles the
smallest attainable permutation p-value is 1/200 — too coarse for a false-discovery-rate correction over 308 genes —
so we used the shuffles to estimate each gene's null mean and standard deviation and turned $I$ into a z-score (with
thousands of cells the null distribution of $I$ is close to normal). 89 of the 308 genes are spatially variable at a
5 % false discovery rate. Moran's I and variance answer different questions. The eight
decoys are among the most *variable* genes (all eight are in the top 30 by variance) but none is spatially variable
(median $I \approx 0$) — a highly-variable-gene list would have kept them, a spatial analysis rejects them. Cell-type
markers are spatial because cell types are arranged in domains: 56 % of them are called, and they fill the top of the
Moran's I list. Of the 21 genes whose true expression differs less than two-fold between cell types, 3 are called:
small but real differences between types are enough to create a weak spatial pattern, so "not a marker" does not mean
"not spatial". All six hypoxia genes are found: each is expressed at a moderate level everywhere (about 0.33 counts per
cell) and rises towards the tumour centre. The exhaustion genes are harder: they are switched on almost only in the T
cells that touch tumour cells (about 0.09 counts per cell on average), so each gene carries a weak signal. Two pass
(`HAVCR2`, `LAG3`) and two do not (`PDCD1`, `TOX`), although all four follow exactly the same rule in the simulator —
gene by gene, detecting a sparse pattern is partly luck. Averaging each programme's genes into one score helps a lot
(last line): the exhaustion score is clearly spatially variable and the hypoxia score far more so than any single
hypoxia gene — the same reason Section 5 works with a score. Note also how small the values of $I$ are (a few
hundredths for most genes): single-cell expression is noisy and cell types are intermixed within domains, so even strongly patterned
genes have modest cell-to-cell autocorrelation, and significance comes from having 4,000 cells.

⚠️ **Significance is not effect size.** With thousands of cells, tiny autocorrelations become significant. Rank genes by
$I$ (or by the fraction of variance explained by space, as SpatialDE does) and look at their maps before interpreting.

✏️ **Your turn.** Recompute `W` with `K_SP = 30` neighbours and re-run 2.2–2.3. Do the values of $I$ grow? Which kind of
pattern (fine-grained or large-scale) does a larger neighbourhood favour?

### 📄 Research connection — Spatial technologies

- **What the field does.** *Spatial Transcriptomics* (Ståhl *et al.* 2016) captured whole-transcriptome RNA on arrays of 100-µm
    barcoded spots; its commercial successor, 10x Visium, uses 55-µm spots. Slide-seq (Rodriques *et al.* 2019) shrank
    the capture units to 10-µm beads, and Stereo-seq and Visium HD reach subcellular bins. In parallel, imaging methods
    descended from single-molecule FISH: MERFISH (Chen *et al.* 2015) reads hundreds to thousands of genes with
    error-robust barcodes over many imaging rounds; commercial platforms (Xenium, CosMx, MERSCOPE) now image hundreds to
    thousands of genes in large sections. Moses & Pachter (2022) chronicle the field in their *Museum of spatial
    transcriptomics*.
- **What this notebook builds.** Both views of one simulated section: single-cell counts (imaging-like) and Visium-like spot mixtures,
    with Moran's I and permutation testing from scratch, graded by gene categories known from the simulator.
- **Where the frontier is.** As of 2025: larger panels and higher resolution blur the line between the two families; 3-D
    reconstruction from serial sections; spatial proteomics and multi-omics on the same section; and computational
    challenges of scale (millions of cells per slide) and of integrating spatial data with dissociated atlases.
- **Read next.** Moses & Pachter 2022 (*Nature Methods*), *Museum of spatial transcriptomics*; Ståhl *et al.* 2016 (*Science*), *Visualization and analysis of gene expression in tissue sections by spatial transcriptomics*.

---
## 3 · Spatial domains

> 📍 **Where we are.** A pathologist outlines regions — tumour, margin, stroma — by their *mixture* of cells, not by
> any single cell. Clustering cells on their own expression finds cell types. To find domains, each cell must be
> described together with its surroundings.

Three representations of each cell, each clustered with k-means into four groups:

1. **Expression only**: the cell's own top 15 principal components (the Module 30 recipe).
2. **Concatenation**: own PCs side by side with the average PCs of its 20 nearest spatial neighbours.
3. **Graph convolution**: repeated neighbour averaging, $H^{(L)} = \hat A^{L} X$, with
   $\hat A = \tilde D^{-1/2}(A + I)\tilde D^{-1/2}$ the normalised adjacency of the spatial graph.

*In words:* each multiplication by $\hat A$ replaces every cell's features by a weighted average over itself and its
neighbours; after $L$ steps a cell's features summarise its $L$-step neighbourhood — its **niche**.

> 🔗 **Builds on** [Module 25](25_graph_neural_networks.ipynb): $\hat A^L X$ is exactly the propagation step of a graph
> convolutional network without weights or nonlinearities (the "SGC" simplification), and over-smoothing (too many
> layers) is the same danger. SpaGCN and related methods learn such filters; BayesSpace uses a Potts-model prior that
> encourages neighbouring spots to share a domain.

In [ ]:
# ── 3.1 Expression-only, concatenated and graph-convolved representations; 🧪 ARI against the true domains ──
Zx = pca(L, n_components=15)[0]                             # own expression (300 real genes only)
def kmeans_labels(F, k=4):
    return KMeans(k, n_init=10, random_state=SEED).fit_predict(F)

nbr20 = NearestNeighbors(n_neighbors=20).fit(xy).kneighbors(xy)[1]
A = sparse.csr_matrix((np.ones(n_cells * K_SP), (np.repeat(np.arange(n_cells), K_SP), nbr6.ravel())), shape=(n_cells, n_cells))
A = A.maximum(A.T) + sparse.eye(n_cells)                    # undirected, with self-loops
d_inv = sparse.diags(1 / np.sqrt(np.asarray(A.sum(axis=1)).ravel()))
A_hat = d_inv @ A @ d_inv

reps = {"expression only": Zx, "own + neighbour average (concatenated)": np.c_[Zx, Zx[nbr20].mean(axis=1)],
        "neighbour average only (20 cells)": Zx[nbr20].mean(axis=1)}
H = Zx.copy()
for layer in range(1, 17):
    H = A_hat @ H
    if layer in (1, 2, 4, 8, 16):
        reps[f"graph convolution, L = {layer}"] = H.copy()
labels_dom, ari_rows = {}, {}
for name, F in reps.items():
    labels_dom[name] = kmeans_labels(F)
    ari_rows[name] = {"ARI vs true domain": adjusted_rand_score(domain, labels_dom[name]),
                      "ARI vs true cell type": adjusted_rand_score(ctype, labels_dom[name])}
print("🧪 k-means (k = 4) on each representation:")
print(pd.DataFrame(ari_rows).T.round(3).to_string())
print(f"(expression only with k = 6 vs true cell type: ARI = {adjusted_rand_score(ctype, kmeans_labels(Zx, 6)):.3f})")

In [ ]:
# ── 3.2 Domain maps ──
dom_names = np.unique(domain)


def name_clusters(lab):
    """Give each cluster the name of the true domain it overlaps most (one-to-one matching, for display only)."""
    table = pd.crosstab(lab, domain).reindex(columns=dom_names, fill_value=0)
    r_, c_ = linear_sum_assignment(-table.to_numpy())
    return np.array([dict(zip(table.index[r_], dom_names[c_]))[c] for c in lab])


fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
panels = [("true domains (hidden)", domain),
          ("k-means on own expression", name_clusters(labels_dom["expression only"])),
          ("k-means after graph convolution (L = 8)", name_clusters(labels_dom["graph convolution, L = 8"]))]
for ax, (title, lab) in zip(axes, panels):
    plot_embedding(xy, lab, ax=ax, s=3, title=title, legend=False)
    ax.set_aspect("equal")
axes[1].legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=4,
               fontsize=7, markerscale=3, frameon=False)
plt.suptitle("SIMULATED: clustering cells vs clustering niches (clusters named after the best-matching domain)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Clustering each cell on its own expression recovers **cell types** (ARI with the true types
1.0 when $k = 6$) but not domains (ARI 0.37): a macrophage in the margin and a macrophage in the stroma look the same.
Concatenating the neighbourhood average to the cell's own PCs barely helps (0.42), because the cell's own identity still
dominates the distances. Describing each cell **only by its surroundings** turns the clustering into a domain finder:
averaging over 20 spatial neighbours gives ARI 0.93, and one to sixteen rounds of graph convolution give 0.75–0.94. The
maps show it: own-expression clusters are salt-and-pepper patterns of cell types, while the smoothed clusters
reproduce the core, the ring of the margin, the epithelial band and the stroma. The ARI peaks at $L = 8$ and starts to
fall at $L = 16$: too much smoothing blurs thin structures such as the margin ring — the over-smoothing of Module 25 in a
spatial guise.

✏️ **Your turn.** Push the graph convolution to $L = 64$ (change the loop limit) and plot the map. Which domain
disappears first, and why is it the thinnest one?

### 📄 Research connection — Spatial patterns and domains

- **What the field does.** Spatially variable gene detection started with Gaussian-process models: SpatialDE (Svensson, Teichmann &
    Stegle 2018) tests whether a spatial covariance kernel explains a gene's variation better than noise alone, and
    reports the fraction of variance explained by space. Domain detection grew from clustering with spatial priors:
    BayesSpace (Zhao *et al.* 2021) uses a Bayesian model with a Potts prior that favours neighbouring spots in the same
    cluster (and can enhance resolution to sub-spot level); SpaGCN (Hu *et al.* 2021) uses a graph convolutional network
    over spots, optionally weighted by histology. Squidpy (Palla *et al.* 2022) and Giotto (Dries *et al.* 2021)
    package spatial graphs, Moran's I, neighbourhood enrichment and more.
- **What this notebook builds.** Moran's I with permutation and FDR control; three cell representations (own, concatenated, graph-
    convolved) clustered and scored against the true domains — the simplest version of the idea behind SpaGCN-type
    methods.
- **Where the frontier is.** Different domain methods often give different answers on the same tissue, and manual annotations are
    themselves uncertain, so evaluation is hard; current work focuses on multi-sample and 3-D domains, on combining histology
    images (and pathology foundation models) with expression, and on statistical tests of spatial patterns that scale
    to millions of cells (GP-based tests are cubic in the number of cells without approximations,
    [Module 29](29_gaussian_processes_bayesian_optimization.ipynb)).
- **Read next.** Svensson, Teichmann & Stegle 2018 (*Nature Methods*), *SpatialDE: identification of spatially variable genes*; Palla *et al.* 2022 (*Nature Methods*), *Squidpy: a scalable framework for spatial omics analysis*.

---
## 4 · Neighbourhoods: which cell types live together?

> 📍 **Where we are.** Domains are regions; now we ask about pairs of cell types. Do T cells sit next to tumour cells
> more often than chance would predict?

**Neighbourhood enrichment** (as in Squidpy): count the edges of the spatial graph that join a cell of type $a$ to a
cell of type $b$, $N_{ab}$. Under the null hypothesis "types are placed at random", shuffling the type labels over the
cells (keeping positions and the graph) gives the distribution of $N_{ab}$, and

$$z_{ab} = \frac{N_{ab} - \operatorname{mean}_{\text{perm}}(N_{ab})}{\operatorname{sd}_{\text{perm}}(N_{ab})} .$$

*In words:* how many standard deviations more (or fewer) $a$–$b$ contacts there are than if the same cells were
scattered randomly. Large positive: co-localised; large negative: segregated.

In [ ]:
# ── 4.1 Neighbourhood-enrichment z-scores by label permutation ──
code_t = np.array([TYPES.index(c) for c in ctype])
src, dst = np.repeat(np.arange(n_cells), K_SP), nbr6.ravel()      # directed 6-NN edges

def contact_counts(lab):
    """Number of spatial-graph edges between every pair of cell types (symmetrised)."""
    N = np.zeros((len(TYPES), len(TYPES)))
    np.add.at(N, (lab[src], lab[dst]), 1)
    return N + N.T


N_obs = contact_counts(code_t)
rng4 = np.random.default_rng(SEED + 4)
N_null = np.array([contact_counts(rng4.permutation(code_t)) for _ in range(300)])
z_nh = (N_obs - N_null.mean(axis=0)) / N_null.std(axis=0)
print("neighbourhood-enrichment z-scores (300 permutations):")
print(pd.DataFrame(z_nh, index=TYPES, columns=TYPES).round(1).to_string())

**The ground truth.** The simulator draws each cell's type from a composition that depends only on its domain. The
compositions below are copied from the simulator's source (hidden truth). If neighbours almost always share a domain,
the expected share of $a$–$b$ contacts is $\sum_d n_d\, p_{da}\, p_{db}$ ($n_d$ = cells in domain $d$, $p_{da}$ = share of
type $a$ in domain $d$), which gives a design-based enrichment ratio to compare with the observed one.

In [ ]:
# ── 4.2 🧪 Observed co-localisation vs the simulated design ──
DESIGN = {  # domain: shares of Tumor, Epithelial, Fibroblast, Endothelial, T cell, Macrophage (from the simulator)
    "tumor core": [0.80, 0.00, 0.03, 0.07, 0.03, 0.07], "invasive margin": [0.25, 0.00, 0.15, 0.05, 0.30, 0.25],
    "epithelium": [0.00, 0.85, 0.10, 0.02, 0.02, 0.01], "stroma": [0.00, 0.02, 0.62, 0.16, 0.08, 0.12]}
n_dom = pd.Series(domain).value_counts()
E_design = sum(n_dom[d] * np.outer(p, p) for d, p in DESIGN.items())
ratio = lambda N: N / np.outer(N.sum(1), N.sum(1)) * N.sum()        # observed / expected under random mixing
iu = np.triu_indices(len(TYPES))
ok = E_design[iu] > 0
lr_design, lr_obs = np.log2(ratio(E_design)[iu][ok]), np.log2(np.maximum(ratio(N_obs)[iu][ok], 1e-3))
print(f"🧪 correlation of log2 enrichment ratios, design vs observed: {np.corrcoef(lr_design, lr_obs)[0, 1]:.3f} "
      f"({ok.sum()} type pairs; the tumour–epithelial pair never meets in the design and is left out)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(np.clip(z_nh, -30, 30), cmap="RdBu_r", vmin=-30, vmax=30)
axes[0].set_xticks(range(len(TYPES))); axes[0].set_xticklabels(TYPES, rotation=45, ha="right", fontsize=8)
axes[0].set_yticks(range(len(TYPES))); axes[0].set_yticklabels(TYPES, fontsize=8)
for i in range(len(TYPES)):
    for j in range(len(TYPES)):
        axes[0].text(j, i, f"{z_nh[i, j]:.0f}", ha="center", va="center", fontsize=7)
axes[0].set_title("enrichment z-scores (clipped at ±30)"); plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[1].scatter(lr_design, lr_obs, s=20, color=CLASS_COLORS[0])
axes[1].plot([-3, 3], [-3, 3], "k--", lw=1)
axes[1].set_xlabel("log2 enrichment expected from the design (hidden)"); axes[1].set_ylabel("log2 enrichment observed")
axes[1].set_title("co-localisation follows the domain design")
plt.suptitle("SIMULATED: which cell types are neighbours?", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The diagonal is strongly positive: cells of the same type cluster (tumour cells in the core,
epithelial cells in their band, fibroblasts in the stroma). Epithelial cells avoid everything else; tumour cells avoid
fibroblasts. The immune cells are mildly enriched next to each other and next to tumour cells — they share the
invasive margin. The observed enrichment ratios match those expected from the simulated domain compositions almost
perfectly (correlation 0.99): in this simulation **all** co-localisation comes from the domain design, nothing more.

⚠️ **Co-localisation has many causes.** Two types can co-localise because they share a domain (as here), because one
attracts the other, because they share blood supply — or because of the tissue's sectioning plane. A z-score says *that*
two types are close, not *why*. And with thousands of cells, every z-score is "significant"; the heatmap's value lies
in the pattern and the sign.

✏️ **Your turn.** Change the null hypothesis: shuffle the type labels only *within* each domain (true or inferred in
Section 3) and recompute `z_nh`. If all co-localisation comes from the domain design, what should the new z-scores look
like? Which pairs, if any, keep a large z-score, and why?

---
## 5 · Communication and niche effects

> 📍 **Where we are.** Co-localisation is about *which* cells are near each other. A **niche effect** is about whether
> a cell's *state* depends on its neighbours — the footprint of cell–cell communication.

The simulator planted one such effect: in T cells, the exhaustion genes (`PDCD1`, `HAVCR2`, `LAG3`, `TOX`) rise with the
number of tumour cells nearby. We pretend not to know the details and test it the way a study would: count the tumour
cells within a radius of 0.04 of every T cell (two to three typical distances between neighbouring cells) and relate
an **exhaustion score** (mean log-normalised expression of the four genes) to that count.

In [ ]:
# ── 5.1 Exhaustion score vs number of tumour neighbours, with controls ──
tree = cKDTree(xy)
n_tumour_nb = np.array([np.sum(code_t[nb] == 0) for nb in tree.query_ball_point(xy, 0.04)])
exh = L[:, np.isin(genes, EXHAUSTION_GENES)].mean(axis=1)
tmark = L[:, np.isin(genes, TISSUE_TYPES["T cell"][1])].mean(axis=1)     # T-cell identity markers
is_T, is_mac = ctype == "T cell", ctype == "Macrophage"

def slope_and_perm(y, x, groups=None, n_perm=999, rng=None):
    """OLS slope of y on x, the mean of the permutation null, a permutation p-value and z = (slope − null mean) / null sd.
    With `groups`, y is shuffled only within each group (the null keeps group differences)."""
    slope = np.polyfit(x, y, 1)[0]
    null = np.empty(n_perm)
    for b in range(n_perm):
        yp = y.copy()
        for gr in (np.unique(groups) if groups is not None else [None]):
            m = np.ones(len(y), bool) if gr is None else groups == gr
            yp[m] = rng.permutation(y[m])
        null[b] = np.polyfit(x, yp, 1)[0]
    centre = null.mean()
    p = (1 + np.sum(np.abs(null - centre) >= abs(slope - centre))) / (n_perm + 1)
    return slope, centre, p, (slope - centre) / null.std()


rng5 = np.random.default_rng(SEED + 5)
dom_inferred = labels_dom["graph convolution, L = 8"]                      # Section 3's domains, no hidden truth
rows = {"T cells: exhaustion score": slope_and_perm(exh[is_T], n_tumour_nb[is_T], rng=rng5),
        "T cells: exhaustion, shuffled within inferred domains":
            slope_and_perm(exh[is_T], n_tumour_nb[is_T], dom_inferred[is_T], rng=rng5),
        "macrophages: exhaustion score (control)": slope_and_perm(exh[is_mac], n_tumour_nb[is_mac], rng=rng5),
        "T cells: T-cell markers (control)": slope_and_perm(tmark[is_T], n_tumour_nb[is_T], rng=rng5)}
print(f"{is_T.sum()} T cells; tumour neighbours per T cell: median {np.median(n_tumour_nb[is_T]):.0f}, "
      f"max {n_tumour_nb[is_T].max()}")
exh_share = counts[:, np.isin(genes, EXHAUSTION_GENES)].sum(axis=1) / counts.sum(axis=1)
print(f"share of the library taken by the exhaustion genes: T cells with ≥ 8 tumour neighbours "
      f"{exh_share[is_T & (n_tumour_nb >= 8)].mean():.1%}, T cells with none {exh_share[is_T & (n_tumour_nb == 0)].mean():.2%}")
print("🧪 slope per extra tumour neighbour and permutation p-value (999 shuffles):")
print(pd.DataFrame(rows, index=["slope", "null mean slope", "p", "z vs null"]).T.round(3).to_string())

In [ ]:
# ── 5.2 The niche effect, seen three ways ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, m, title in [(axes[0], is_T, "T cells"), (axes[1], is_mac, "macrophages (control)")]:
    ax.scatter(n_tumour_nb[m] + np.random.default_rng(1).uniform(-0.3, 0.3, m.sum()), exh[m], s=5, alpha=0.4, color="0.4")
    xs = np.arange(0, 21)
    means = [exh[m & (n_tumour_nb == v)].mean() if np.sum(m & (n_tumour_nb == v)) >= 5 else np.nan for v in xs]
    ax.plot(xs, means, "o-", color=CLASS_COLORS[1], ms=4, label="mean per count")
    ax.set_xlabel("tumour cells within radius 0.04"); ax.set_title(title); ax.set_xlim(-1, 21)
axes[0].axvline(8, color="k", ls=":", lw=1); axes[0].text(8.3, exh[is_T].min(), "simulator's cap\n(hidden)", fontsize=7)
axes[0].set_ylabel("exhaustion score"); axes[0].legend(fontsize=7)
axes[2].scatter(*xy[code_t == 0].T, s=1, color="0.8", label="tumour cells")
o = np.argsort(exh[is_T])
sc = axes[2].scatter(*xy[is_T][o].T, c=exh[is_T][o], s=8, cmap="viridis")
plt.colorbar(sc, ax=axes[2], shrink=0.8, label="exhaustion score")
axes[2].set_aspect("equal"); axes[2].set_xticks([]); axes[2].set_yticks([]); axes[2].set_title("T cells on the map")
plt.suptitle("SIMULATED niche effect: T-cell exhaustion grows with tumour contact", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In T cells, every additional tumour neighbour raises the exhaustion score by about 0.22
log units, far outside the permutation null (p = 0.001, the smallest possible with 999 shuffles; the slope is 16
null standard deviations from the null mean). Macrophages, which do not respond in the simulator, show at most a tiny
slope (0.007, about 3 % of the T-cell slope). Its p-value of 0.03 is borderline: with no mechanism for it in the
simulator, it is the kind of chance result that turns up now and then when several tests are run — a reminder to weigh
effect sizes and to correct for multiple testing. The T cells' own identity markers show no significant change (slope
−0.013, p = 0.17). A small decrease is in fact expected: the simulator, like sequencing, gives each cell a library of
fixed size, so when the exhaustion genes take a larger share (up to about 2 % of the library in T cells with eight or
more tumour neighbours), every other gene's share shrinks a little — a **compositional effect** of a fixed-size library
that normalisation does not undo (the trap of the composition analysis of Module 30). Here it is too small to detect;
with a strong programme it would make a control look "significant". The binned means rise roughly linearly and then level off near the hidden cap of 8
tumour neighbours, beyond which the simulator adds no more exhaustion. On the map, the most exhausted T cells sit
inside and at the edge of the tumour.

**Is it the niche or the domain?** T cells near tumour cells are mostly in the margin and core, so a domain-wide effect
(e.g. "T cells in tumours are exhausted, whatever their neighbours") would also produce a slope. The second row shuffles
the scores only *within* the domains inferred in Section 3. The observed slope is of course unchanged, but the null
changes: because shuffling keeps each domain's scores in that domain, domain differences alone already produce an
average slope of 0.17. Most of the raw association could thus be explained by "T cells in tumour domains are
exhausted". The observed 0.22 is nevertheless well outside this conditional null (z ≈ 8, p = 0.001): *within* a domain,
T cells with more tumour neighbours are more exhausted — the dose–response the simulator planted. That is the logic
of a **conditional permutation test**: the null hypothesis keeps the confounder fixed.

### Ligand–receptor analysis: what it can and cannot show

Communication between cells is usually mediated by a **ligand** (secreted or on the membrane of the sender) binding a
**receptor** on the receiver. Tools such as CellPhoneDB, CellChat (Jin *et al.* 2021) and their spatial extensions
score known ligand–receptor pairs by co-expression: high ligand in one cell type and high receptor in another —
restricted, in spatial data, to cells that are neighbours. A simple spatial score for a pair ($\ell$, $r$) is the mean
over spatial-graph edges of $x_{i\ell}\,x_{jr}$, compared with a null distribution. We score two "pairs":

- `EPCAM` (in tumour cells) → `PDCD1` (in T cells): the simulator *does* make T cells respond to tumour neighbours;
- `DCN` (in fibroblasts) → `CDH5` (in endothelial cells): **no** interaction exists in the simulator.

Neither is a real ligand–receptor pair — we only need one gene expressed by each partner to show how the statistics
behave.

In [ ]:
# ── 5.3 A spatial co-expression score with two different null hypotheses ──
def coexpression(lig, rec, perm=None):
    """Mean over spatial-graph edges i → j of x_i(lig) · x_j(rec); `perm` moves whole cells to other positions."""
    a, b = L[:, gidx[lig]], L[:, gidx[rec]]
    if perm is not None:
        a, b = a[perm], b[perm]
    return np.mean(a[src] * b[dst])


def perm_within_types(rng):
    """Shuffle cells only among cells of the same type: keeps who-is-next-to-which-type, breaks within-type niches."""
    perm = np.arange(n_cells)
    for t in range(len(TYPES)):
        members = np.flatnonzero(code_t == t)
        perm[members] = rng.permutation(members)
    return perm


rng53 = np.random.default_rng(SEED + 53)
rows = {}
for lig, rec, truth in [("EPCAM", "PDCD1", "yes (planted niche effect)"), ("DCN", "CDH5", "no")]:
    obs = coexpression(lig, rec)
    null_any = np.array([coexpression(lig, rec, rng53.permutation(n_cells)) for _ in range(500)])
    null_type = np.array([coexpression(lig, rec, perm_within_types(rng53)) for _ in range(500)])
    rows[f"{lig} → {rec}"] = {"interaction in simulator": truth,
                              "z vs random positions": round((obs - null_any.mean()) / null_any.std(), 1),
                              "z vs shuffling within cell types": round((obs - null_type.mean()) / null_type.std(), 1)}
print("🧪 co-expression scores of neighbouring cells (500 permutations each):")
print(pd.DataFrame(rows).T.to_string())

🔍 **What just happened?** Against the usual null — cells moved to random positions — **both** pairs are clearly
"significant" (z ≈ 7 and z ≈ 5.5), although only the first corresponds to anything happening in the simulated tissue.
The fibroblast–endothelial "interaction" is pure **co-localisation**: both types live in the stroma, so their genes are
co-expressed in neighbouring cells. The second null keeps every position occupied by a cell of the same type and only
shuffles cells *within* types. It asks a sharper question: "given who is next to whom, does the receiver's expression
depend on its particular neighbours?" Now the fake pair vanishes (z ≈ 0.5), while the planted niche effect remains
(z ≈ 6).

⚠️ **Co-localisation ≠ signalling.** Even the surviving signal is a correlation: it shows that T cells with more tumour
neighbours express more `PDCD1`, not that a particular ligand caused it. Real ligand–receptor analyses add further
caveats: mRNA levels are poor proxies for protein abundance and binding; many ligands share receptors; secreted ligands
act beyond the nearest neighbours; and databases of pairs are incomplete. Establishing communication needs perturbation
(blocking the ligand or deleting the receptor) — the subject of [Module 41](41_perturbation_causal_inference.ipynb).

✏️ **Your turn.** The radius 0.04 used in 5.1 happens to be the simulator's (hidden) interaction radius. Recompute
`n_tumour_nb` with radii 0.02 and 0.08. How does the slope change, and how would you choose a radius on real data?

---
## 6 · Deconvolution of spots

> 📍 **Where we are.** Back to the sequencing view: each Visium-like spot contains several cells. Deconvolution
> estimates the cell-type composition of every spot from its counts and a **reference** — the average expression
> profile of each cell type, usually taken from a single-cell RNA-seq atlas of the same tissue.

**The model.** Let $y_{sg}$ be the counts of gene $g$ in spot $s$, and $r_{tg}$ the reference profile of type $t$ (the
share of a type-$t$ cell's transcripts that come from gene $g$; each profile sums to 1). If a spot contains
$h_{st}$ transcripts' worth of type $t$,

$$\mathbb E[y_{sg}] = \sum_t h_{st}\, r_{tg}, \qquad h_{st} \ge 0, \qquad \text{proportion}_{st} = \frac{h_{st}}{\sum_{t'} h_{st'}} .$$

*In words:* a spot's expected counts are a non-negative mixture of the cell-type profiles. Two ways to fit $h$:

1. **Non-negative least squares (NNLS)**: minimise $\sum_g (y_{sg} - \sum_t h_{st} r_{tg})^2$ subject to $h \ge 0$
   (`scipy.optimize.nnls`). Least squares with a constraint — the constrained regression of
   [Module 07](07_ridge_lasso_sparse_regression.ipynb).
2. **Poisson likelihood**: counts are Poisson around the mixture; maximise $\sum_g [y_{sg}\log\mu_{sg} - \mu_{sg}]$.
   Lee & Seung (1999) introduced multiplicative updates for exactly this objective, and proved their monotonicity in
   Lee & Seung (2001):

$$h_{st} \leftarrow h_{st}\;\frac{\sum_g r_{tg}\, y_{sg}/\mu_{sg}}{\sum_g r_{tg}}, \qquad \mu_{sg} = \sum_{t'} h_{st'} r_{t'g},$$

   each update never decreases the likelihood and keeps $h \ge 0$ automatically. *In words:* each type's amount is multiplied by
   the average ratio "observed / predicted" over that type's genes: if the spot has more of type $t$'s genes than
   predicted, $h_{st}$ grows. This is non-negative matrix factorisation with the reference held fixed; with fixed
   profiles it is also exactly the EM algorithm for a Poisson mixture (known in imaging as Richardson–Lucy
   deconvolution), which is why each step can only improve the likelihood (🔗 [Module 13](13_mixture_models_em.ipynb)).
   This likelihood is the core of
   RCTD's likelihood (Cable *et al.* 2022), which adds overdispersion and platform effects; cell2location
   (Kleshchevnikov *et al.* 2022) adds a full Bayesian model.

> 🔗 **Builds on** [Module 06](06_linear_regression_gwas.ipynb) (least squares) and [Module 05](05_likelihood_mle_map.ipynb)
> (maximum likelihood): the same mixture model fitted with two different noise assumptions.

In [ ]:
# ── 6.1 NNLS and Poisson deconvolution; 🧪 against the true spot compositions ──
R_ref = tissue["reference"].to_numpy()                      # types × genes, each row sums to 1
has_cells = spots.n_cells.to_numpy() > 0
Y = spot_counts[has_cells].astype(float)
P_true = tissue["spot_proportions"].to_numpy()[has_cells]


def poisson_deconvolve(Y, R, n_iter=500):
    """Poisson-likelihood mixture weights h ≥ 0 by Lee–Seung multiplicative updates (reference R held fixed)."""
    H = np.full((len(Y), len(R)), 1.0) * Y.sum(axis=1, keepdims=True) / len(R)
    for _ in range(n_iter):
        mu = H @ R
        H *= (Y / np.maximum(mu, 1e-12)) @ R.T / R.sum(axis=1)
    return H


def to_props(H):
    return H / H.sum(axis=1, keepdims=True)


t0 = time.time()
P_nnls = to_props(np.array([nnls(R_ref.T, y)[0] for y in Y]))
P_pois = to_props(poisson_deconvolve(Y, R_ref))
print(f"{len(Y)} spots deconvolved in {time.time() - t0:.1f} s")
rows = {}
for name, P in [("NNLS", P_nnls), ("Poisson (multiplicative updates)", P_pois)]:
    rows[name] = {"RMSE": np.sqrt(np.mean((P - P_true) ** 2))}
    rows[name] |= {t: np.corrcoef(P[:, j], P_true[:, j])[0, 1] for j, t in enumerate(TYPES)}
print("🧪 RMSE of the proportions and per-type correlation with the true composition:")
print(pd.DataFrame(rows).T.round(3).to_string())

In [ ]:
# ── 6.2 True vs estimated compositions ──
sxy = spots[["x", "y"]].to_numpy()[has_cells]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, P, title in [(axes[0], P_true, "true tumour share per spot (hidden)"), (axes[1], P_pois, "Poisson estimate")]:
    sc = ax.scatter(*sxy.T, c=P[:, 0], s=22, cmap="Reds", vmin=0, vmax=1, marker="h")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
plt.colorbar(sc, ax=axes[1], shrink=0.8)
for j, t in enumerate(TYPES):
    axes[2].scatter(P_true[:, j], P_pois[:, j], s=6, alpha=0.5, color=plt.get_cmap("tab10")(j), label=t)
axes[2].plot([0, 1], [0, 1], "k--", lw=1); axes[2].legend(fontsize=6.5, markerscale=2)
axes[2].set_xlabel("true proportion (hidden)"); axes[2].set_ylabel("estimated proportion (Poisson)")
axes[2].set_title("all spots, all types")
plt.suptitle("SIMULATED Visium-like spots: deconvolution against the truth", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2b 🧪 Does every multiplicative update increase the Poisson log-likelihood? ──
def poisson_loglik(Y, H, R):
    """Poisson log-likelihood of the spot counts, up to a constant: Σ y log μ − μ."""
    mu = np.maximum(H @ R, 1e-12)
    return float(np.sum(Y * np.log(mu) - mu))


H_run = np.full((len(Y), len(R_ref)), 1.0) * Y.sum(axis=1, keepdims=True) / len(R_ref)
trace = []
for it in range(200):
    H_run *= (Y / np.maximum(H_run @ R_ref, 1e-12)) @ R_ref.T / R_ref.sum(axis=1)
    trace.append(poisson_loglik(Y, H_run, R_ref))
W_ls = np.array([nnls(R_ref.T, y)[0] for y in Y])                # the NNLS fit, scored with the same likelihood
print(f"log-likelihood after 1, 10, 50, 200 updates: {trace[0]:.0f}, {trace[9]:.0f}, {trace[49]:.0f}, {trace[-1]:.0f}")
print(f"🧪 decreases in 200 updates: {int(np.sum(np.diff(trace) < -1e-6))}; "
      f"NNLS solution under the Poisson likelihood: {poisson_loglik(Y, W_ls, R_ref):.0f}")

🔍 **What just happened?** The log-likelihood rises at every one of the 200 multiplicative updates, as the EM argument
guarantees, and it ends above the likelihood of the NNLS solution — least squares optimises a different
criterion. Both methods recover the compositions well — the estimated tumour map is almost indistinguishable from the
true one — but the Poisson fit is clearly better (RMSE about 0.048 against 0.073 for NNLS),
especially for the rarer, less distinctive types (endothelial cells: correlation 0.93 against 0.76). Least squares
treats an error of 10 counts the same in a gene with 1,000 counts as in a gene with 5, so highly expressed genes
dominate the fit; the Poisson likelihood weights each gene by its expected noise. The remaining errors come from
counting noise in spots of only a few cells and, to a lesser extent, from cell states missing in the reference
(hypoxic tumour cells, exhausted T cells), which shift those cells' profiles slightly.

### What if the reference is incomplete?

A reference built from dissociated cells often misses a type: fragile cells (neurons, adipocytes) do not survive
dissociation, or a type is specific to the tissue under study. We remove macrophages from the reference and deconvolve
again.

In [ ]:
# ── 6.3 🧪 A cell type missing from the reference ──
mac = TYPES.index("Macrophage")
keep = [j for j in range(len(TYPES)) if j != mac]
H_miss = poisson_deconvolve(Y, R_ref[keep])
P_miss = to_props(H_miss)
print(f"RMSE over the five remaining types: full reference {np.sqrt(np.mean((P_pois[:, keep] - P_true[:, keep]) ** 2)):.3f}, "
      f"macrophages missing {np.sqrt(np.mean((P_miss - P_true[:, keep]) ** 2)):.3f}")
err = P_miss - P_true[:, keep]
print("where the macrophage share goes — mean error of each type, and its correlation with the true macrophage share:")
corr_mac = [np.corrcoef(err[:, k], P_true[:, mac])[0, 1] for k in range(len(keep))]
print(pd.DataFrame({"mean error": err.mean(0), "corr with true macrophage share": corr_mac},
                   index=[TYPES[j] for j in keep]).round(3).to_string())
mu = H_miss @ R_ref[keep]
resid = (Y - mu) / np.sqrt(mu + 1e-9)                       # Pearson residuals of the fit
mac_genes = var.marker_of.to_numpy(str) == "Macrophage"
res_score = resid[:, mac_genes].mean(axis=1)
print(f"🧪 diagnostic: mean residual of the macrophage marker genes vs true macrophage share: "
      f"r = {np.corrcoef(res_score, P_true[:, mac])[0, 1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(P_true[:, mac], err[:, keep.index(TYPES.index("Epithelial"))], s=8, color=CLASS_COLORS[2])
axes[0].set_xlabel("true macrophage share (hidden)"); axes[0].set_ylabel("error of the 'epithelial' estimate")
axes[0].set_title("missing macrophages are absorbed by another type")
axes[1].scatter(P_true[:, mac], res_score, s=8, color=CLASS_COLORS[3])
axes[1].set_xlabel("true macrophage share (hidden)"); axes[1].set_ylabel("mean residual of macrophage markers")
axes[1].set_title("…but the residuals give it away")
plt.suptitle("SIMULATED: deconvolution with an incomplete reference", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without macrophages in the reference, the model must explain their transcripts with the types
it has. The error of the remaining five types rises (RMSE 0.049 → 0.084), and most of the macrophage share is handed to
**epithelial cells** — a type that does not even occur in the tumour region. The epithelial error tracks the true
macrophage share closely (correlation 0.87). A biologist reading the output would conclude that the invasive margin
contains normal epithelium. The **residuals** reveal the problem: the macrophage marker genes are systematically
under-predicted exactly where macrophages are (correlation 0.89). Checking residuals per gene and per spot is the
practical defence against an incomplete reference.

✏️ **Your turn.** Simulate a **platform effect** — the atlas and the spatial assay capture genes with different
efficiencies — by multiplying each gene of `R_ref` by `np.exp(rng.normal(0, 0.5))`, renormalising the rows, and
deconvolving again. How much does the RMSE grow? RCTD estimates one such factor per gene; how could you estimate it from
the spot data?

### 📄 Research connection — Deconvolution and cell–cell communication

- **What the field does.** Spot deconvolution methods differ mainly in the noise model and the prior: RCTD (Cable *et al.* 2022) fits
    a Poisson–lognormal mixture with per-gene platform effects; cell2location (Kleshchevnikov *et al.* 2022) is a
    hierarchical Bayesian negative-binomial model that shares information across spots; Tangram maps single cells onto
    space directly; non-negative matrix factorisation (Lee & Seung 1999) is the common mathematical ancestor. For
    communication, CellPhoneDB and CellChat (Jin *et al.* 2021) curate ligand–receptor databases and score pairs by
    co-expression across cell types; spatial methods restrict the scores to neighbours and model distance.
- **What this notebook builds.** NNLS and a Poisson multiplicative-update deconvolution graded against true spot compositions; the bias
    from a missing reference type and a residual diagnostic; a niche-effect regression with conditional permutation;
    a co-expression score showing how the choice of null separates co-localisation from niche dependence.
- **Where the frontier is.** With subcellular platforms (Visium HD, Stereo-seq, imaging), deconvolution gives way to segmentation
    and to "bin-to-cell" assignment; with spot data, current models add spatial smoothness priors and estimate
    continuous cell states, not only types. For communication, the open problem is validation: inferred interactions
    are rarely tested by perturbation, and benchmarks with known ground truth are scarce — simulations like ours are one
    of the few places where the answer is known.
- **Read next.** Cable *et al.* 2022 (*Nature Biotechnology*), *Robust decomposition of cell type mixtures in spatial transcriptomics*; Kleshchevnikov *et al.* 2022 (*Nature Biotechnology*), *Cell2location maps fine-grained cell types in spatial transcriptomics*.

---
## 7 · Segmentation and resolution

> 📍 **Where we are.** Imaging platforms locate individual molecules, and an algorithm must decide which cell each
> molecule belongs to — **segmentation**, usually from a nuclear or membrane stain. Cells overlap, membranes are faint,
> and molecules near a boundary are easily assigned to the neighbour. What does that do to the analysis?

We simulate the most common error, **spillover**: every cell receives a fraction $f$ of the transcripts of its nearest
neighbour (each molecule independently, `rng.binomial`). We then annotate each cell by maximum likelihood under a
multinomial model with the reference profiles of Section 6,
$\hat t = \arg\max_t \sum_g y_g \log r_{tg}$, leaving out the hypoxia and exhaustion genes (cell *states* the reference
does not describe), and we re-run the niche analysis of Section 5 with a gene that T cells never express: the tumour
marker set.

In [ ]:
# ── 7.1 Spillover from neighbouring cells: mis-annotation and a spurious niche effect ──
nn1 = nbr6[:, 0]                                            # each cell's nearest neighbour
state_genes = np.isin(genes, HYPOXIA_GENES + EXHAUSTION_GENES)
R_id = R_ref[:, ~state_genes] / R_ref[:, ~state_genes].sum(axis=1, keepdims=True)
tumour_markers = np.isin(genes, TISSUE_TYPES["Tumor"][1])
nb_is_tumour = code_t[nn1] == 0
rng7 = np.random.default_rng(SEED + 7)
rows = {}
for f in [0.0, 0.1, 0.2, 0.3, 0.5]:
    C = counts + rng7.binomial(counts[nn1], f)              # transcripts leaking in from the neighbour
    pred = np.argmax(C[:, ~state_genes] @ np.log(R_id).T, axis=1)
    tm = log_normalize(C)[:, tumour_markers].mean(axis=1)
    edge_T = is_T & nb_is_tumour
    rows[f"f = {f}"] = {"accuracy, all cells": np.mean(pred == code_t),
                        "accuracy, T cells": np.mean(pred[is_T] == code_t[is_T]),
                        "T cells next to a tumour cell called correctly": np.mean(pred[edge_T] == code_t[edge_T]),
                        "slope: tumour markers in T cells vs tumour neighbours": np.polyfit(n_tumour_nb[is_T], tm[is_T], 1)[0]}
seg = pd.DataFrame(rows).T
print(f"🧪 {np.sum(is_T & nb_is_tumour)} T cells have a tumour cell as nearest neighbour; true slope of the last column = 0")
print(seg.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
fs = [0.0, 0.1, 0.2, 0.3, 0.5]
for col, c in zip(seg.columns[:3], CLASS_COLORS):
    axes[0].plot(fs, seg[col], "o-", color=c, label=col)
axes[0].set_xlabel("spillover fraction f"); axes[0].set_ylabel("annotation accuracy"); axes[0].legend(fontsize=7)
axes[0].set_title("mis-annotation grows with spillover")
axes[1].plot(fs, seg.iloc[:, 3], "o-", color=CLASS_COLORS[1]); axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set_xlabel("spillover fraction f"); axes[1].set_ylabel("slope per tumour neighbour")
axes[1].set_title("a niche effect that does not exist")
plt.suptitle("SIMULATED segmentation errors", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With perfect segmentation ($f = 0$) the reference-based annotation is essentially perfect
(100 % of cells, 99.7 % of T cells). As spillover grows, overall accuracy declines only slowly, because most cells
are surrounded by their own type — but cells at **boundaries between types** suffer. T cells are a minority type, so
their nearest neighbour is usually a cell of another type: they are called correctly 99.7 % of the time at $f = 0$,
89 % at $f = 0.3$ and 70 % at $f = 0.5$ (those next to a tumour cell: 100 %, 90 % and 72 %). Worse, spillover creates a **niche effect out of nothing**: T cells express no tumour markers in the
simulation, yet the tumour-marker score of T cells now rises with the number of tumour neighbours, and the slope grows
steadily with $f$. In a real study this would read as "T cells near tumours acquire epithelial features". Spillover
mimics exactly the kind of neighbour-dependent expression that Section 5 was looking for — which is why imaging studies
check niche effects against genes the receiving cell type should never express, and why segmentation is an active
research area.

✏️ **Your turn.** Put the state genes back into the annotation (use `R_ref` and all genes in `pred`). Does the accuracy
at $f = 0$ change? In this simulation the state programmes take at most a few per cent of a cell's library; how strong
would a state programme have to be before a reference built from "average" cells starts to misclassify cells in that
state? (Try multiplying the exhaustion genes' counts in T cells by 20.)

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above used one simulated square with 4,000 cells and an answer key. Here is what
> changes on a real slide — no downloads, a map for your first real analysis.

**Scale and formats.** A Visium slide has about 5,000 spots per capture area; imaging platforms measure hundreds of
thousands to millions of cells per section, with the coordinates of every molecule. Data come with a histology image
(H&E or immunofluorescence) aligned to the coordinates. The scverse ecosystem stores them as
[SpatialData](https://spatialdata.scverse.org/) objects (images, shapes, points and tables together); Seurat and Giotto
are the R counterparts.

**Public data to practise on.** [10x Genomics datasets](https://www.10xgenomics.com/datasets) (Visium, Visium HD and
Xenium sections of mouse brain, human cancers and more), the
[Allen Brain Cell Atlas](https://portal.brain-map.org/) (whole-brain MERFISH), and tumour atlases of the
[Human Tumor Atlas Network](https://humantumoratlas.org/). [CELLxGENE](https://cellxgene.cziscience.com/) hosts some
spatial collections next to the dissociated references needed for deconvolution.

**Production tools.** [Squidpy](https://squidpy.readthedocs.io/) (spatial graphs, Moran's I, neighbourhood enrichment,
ligand–receptor scoring), [Giotto](https://github.com/drieslab/Giotto), SpaGCN and BayesSpace (domains), SpatialDE
(spatially variable genes), [cell2location](https://cell2location.readthedocs.io/) and RCTD (in
[spacexr](https://github.com/dmcable/spacexr)) for deconvolution, [Tangram](https://github.com/broadinstitute/Tangram)
for mapping single cells onto space, CellChat and CellPhoneDB for communication; segmentation tools such as Cellpose and
Baysor for imaging data.

**What goes wrong first.**

1. **Tissue and capture artefacts.** Folds, tears, bubbles and edge effects create spatial patterns that are not
   biology; lateral diffusion of mRNA blurs sharp boundaries. Always look at the histology and at per-spot QC maps
   (total counts, detected genes, mitochondrial fraction) before testing anything spatial.
2. **Reference mismatch.** Deconvolution and annotation depend on a reference from another assay, often from other
   patients. Missing types, cell states and platform effects bias the results (Section 6) — check residuals.
3. **One section is one sample.** Thousands of spots from one slide are not independent replicates of anything except
   that slide. Claims about patients or conditions need several sections from several individuals, with the
   individual as the unit of replication — the pseudoreplication lesson of Module 30 again.

**Histology as data.** Every Visium and many imaging data sets come with an image of the same section. Models that
predict expression from H&E images, and pathology foundation models trained on very large collections of slides, are an
active area; they promise spatial insight from routine clinical slides, but their predictions must be validated against
measured expression.

## 📌 Key takeaways

- **Two families of spatial transcriptomics**: imaging-based (single cells, targeted panels, segmentation errors) and
  sequencing-based (whole transcriptome, spots that mix several cells). Our simulator gave both views of one section.
- **Spatially variable ≠ highly variable.** Moran's I (a correlation between each cell and its neighbours' average),
  tested by permutation with FDR control, rejected spatially random but highly variable decoy genes that a variance
  ranking kept; sparse programmes are detected far more reliably as summed scores than gene by gene.
- **Domains are niches, not cells.** Clustering own expression found cell types (ARI 0.37 with domains); clustering
  neighbourhood-averaged or graph-convolved features ($\hat A^L X$) found the domains (ARI up to 0.94).
- **Neighbourhood enrichment** quantifies co-localisation by label permutation; here it matched the simulated domain
  design (r = 0.99) — co-localisation alone says nothing about mechanism.
- **Niche effects** are tested within a receiver type, with controls and **conditional permutations** that hold
  confounders (domain, cell-type layout) fixed; the planted T-cell exhaustion effect survived, a co-localisation-only
  "ligand–receptor" pair did not.
- **Deconvolution** fits spots as non-negative mixtures of reference profiles; a Poisson likelihood beat least squares,
  and a missing reference type was silently absorbed by another type — but showed up in the residuals.
- **Segmentation errors** mis-annotate cells at boundaries and can fabricate niche effects; check with genes the
  receiver should never express.

---
## ✅ Checkpoint

1. Compute Moran's I by hand for four cells on a path 1–2–3–4 with row-standardised weights and values x = (2, 2, 0, 0).
   <details><summary>Answer</summary>Mean 1, so z = (1, 1, −1, −1) and Σz² = 4. Neighbour averages: (Wz)₁ = z₂ = 1, (Wz)₂ = (z₁ + z₃)/2 = 0, (Wz)₃ = (z₂ + z₄)/2 = 0, (Wz)₄ = z₃ = −1. Σ zᵢ(Wz)ᵢ = 1 + 0 + 0 + 1 = 2, so I = 2/4 = 0.5 — the same as the toy patch [1, 1, 0, 0] in 2.1 (Moran's I does not depend on the scale of x).</details>
2. Why are spatial patterns tested by permutation rather than with a standard formula?
   <details><summary>Answer</summary>Under the null hypothesis that position does not matter, every assignment of the observed values to the positions is equally likely, so shuffling gives the null distribution for the actual data, graph and gene without any distributional assumption. Analytic formulas for the mean and variance of I exist (under normality or randomisation assumptions) and work well for many cells; permutations check them, and here we used the permutations to calibrate z-scores.</details>
3. A gene is among the 30 most variable genes but has Moran's I ≈ 0. Give a biological example of such a gene.
   <details><summary>Answer</summary>A cell-cycle gene in a tissue where dividing cells are scattered at random, or a gene responding to a stochastic process in individual cells (bursting, stress) — large cell-to-cell variation without spatial organisation, like our decoys.</details>
4. Why does clustering expression smoothed over spatial neighbours find domains, while clustering own expression finds cell types?
   <details><summary>Answer</summary>Own expression is dominated by cell identity; cells of the same type in different domains look alike. After neighbour averaging, a cell's features describe the mixture of types around it, and domains are defined by such mixtures (a margin is 'tumour + T cells + macrophages').</details>
5. What does spot deconvolution assume, and name two ways the assumption fails on real data.
   <details><summary>Answer</summary>That each spot's expected counts are a non-negative combination of fixed cell-type profiles from the reference. It fails when a type is missing from the reference, when cell states (hypoxia, activation) change profiles, when the reference and spatial platforms capture genes with different efficiencies (platform effects), or when mRNA diffuses between spots.</details>
6. Two cell types have a neighbourhood-enrichment z-score of 25. Does that prove that they communicate? What analysis would get closer?
   <details><summary>Answer</summary>No: co-localisation can come from shared domains, shared blood supply or tissue architecture. Closer evidence is a dose–response of the receiver's state with the number of sender neighbours *within* the receiver type and domain (a conditional permutation, Section 5) — and ultimately a perturbation of the putative signal.</details>
7. Reading code: in `poisson_deconvolve`, why does `H` stay non-negative without any explicit constraint?
   <details><summary>Answer</summary>Each update multiplies H by a ratio of sums of non-negative numbers (reference values, counts and positive predicted means), so a positive starting value stays positive — the multiplicative form enforces the constraint automatically.</details>
8. In Section 7, the tumour-marker score of T cells rose with the number of tumour neighbours although T cells never express tumour markers. Explain the mechanism and one safeguard.
   <details><summary>Answer</summary>Spillover: transcripts of neighbouring tumour cells are assigned to the T cell, more so the more tumour cells surround it. Safeguards: test niche effects with genes the receiver cannot express (negative controls), use membrane stains and better segmentation, or model spillover explicitly.</details>

## 🧠 Questions & discussion

1. Our section had sharp domain boundaries and no tissue artefacts. List three features of a real tumour section that would make each of Sections 2–4 harder, and how you would check for them.
2. Spatial data from one patient contain thousands of spots. What can and cannot be concluded about a disease from one section, and how many patients would you want for a claim such as 'exhausted T cells accumulate at the margin in responders'?
3. Imaging panels contain a few hundred genes chosen in advance. How might the choice of panel bias which spatial patterns and cell types a study can discover?
4. A paper reports 50 'significant' ligand–receptor interactions from one Visium section. Which controls from Section 5 would you ask for?
5. Spatial atlases of tumours are being linked to clinical outcomes and, increasingly, to models that predict expression from routine histology. What could go wrong if such models are trained mostly on samples from a few hospitals or populations, and who would bear the consequences?

## 🚀 Bonus — going further

- **Geary's C.** Implement Geary's C, $C = \frac{(n-1)\sum_{ij} w_{ij}(x_i - x_j)^2}{2\,(\sum_{ij} w_{ij})\,\sum_i z_i^2}$, which is more sensitive to local differences, and compare its ranking of genes with Moran's I.
- **A Gaussian-process test for spatial variation** ([Module 29](29_gaussian_processes_bayesian_optimization.ipynb)): on a subsample of 500 cells, fit a GP with a squared-exponential kernel over (x, y) plus noise to one gene and compare its marginal likelihood with a noise-only model — the idea of SpatialDE.
- **Spot-level domains.** Run the domain analysis of Section 3 on the spot data (deconvolved proportions or normalised spot counts) instead of single cells. How does the ARI change with the coarser resolution?
- **3-D tissue.** Simulate several sections with `simulate_spatial_tissue` and different seeds as a 'stack' (offset in z); build a 3-D k-NN graph and compare Moran's I in 2-D and 3-D.
- **Try the real thing.** Install Squidpy, load one of its example data sets, and reproduce Moran's I, neighbourhood enrichment and a co-occurrence analysis; compare its ligand–receptor results with the cautions of Section 5.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 33 — Spatial omics: tissues as data       Date:
Biological question: where are cell types and states in a tumour, which domains exist, does contact with tumour
                     cells change T cells, and what is inside each spot?
Data (SIMULATED): simulate_spatial_tissue (4,000 cells × 300 genes, 449 spots; truth: domain, cell type, hypoxia
                  gradient, exhaustion vs tumour neighbours, spot proportions) + 8 spatially random decoy genes
Method: Moran's I + permutation + BH; k-means on own / neighbour-averaged / graph-convolved PCs; neighbourhood
        enrichment; slope + conditional permutation; co-expression scores with two nulls; NNLS and Poisson
        deconvolution; spillover simulation
Key equation: I = Σ z_i (Wz)_i / Σ z_i²;  H^(L) = Â^L X;  h ← h · [R (y/μ)] / R·1
Key parameters: k of the spatial graph; number of permutations; L; radius for neighbours; reference used
Ground-truth check result: decoys significant …; ARI domains (own vs smoothed) …; design correlation …;
                           deconvolution RMSE (NNLS / Poisson / missing type) …; spillover slope …
What changed when I modified …: (e.g. K_SP, L, radius, platform effect)
One failure mode: (e.g. co-localisation mistaken for signalling; missing reference type; spillover)
Connection to research (paper/tool): Squidpy, SpatialDE, SpaGCN/BayesSpace, RCTD/cell2location, CellChat
```

## 📚 References

- Ståhl P. L. *et al.* (2016) Visualization and analysis of gene expression in tissue sections by spatial transcriptomics. *Science* 353:78–82.
- Chen K. H., Boettiger A. N., Moffitt J. R., Wang S., Zhuang X. (2015) Spatially resolved, highly multiplexed RNA profiling in single cells. *Science* 348:aaa6090.
- Rodriques S. G. *et al.* (2019) Slide-seq: a scalable technology for measuring genome-wide expression at high spatial resolution. *Science* 363:1463–1467.
- Moses L., Pachter L. (2022) Museum of spatial transcriptomics. *Nature Methods* 19:534–546.
- Marx V. (2021) Method of the Year: spatially resolved transcriptomics. *Nature Methods* 18:9–14.
- Moran P. A. P. (1950) Notes on continuous stochastic phenomena. *Biometrika* 37:17–23.
- Svensson V., Teichmann S. A., Stegle O. (2018) SpatialDE: identification of spatially variable genes. *Nature Methods* 15:343–346.
- Hu J. *et al.* (2021) SpaGCN: integrating gene expression, spatial location and histology to identify spatial domains and spatially variable genes by graph convolutional network. *Nature Methods* 18:1342–1351.
- Zhao E. *et al.* (2021) Spatial transcriptomics at subspot resolution with BayesSpace. *Nature Biotechnology* 39:1375–1384.
- Palla G. *et al.* (2022) Squidpy: a scalable framework for spatial omics analysis. *Nature Methods* 19:171–178.
- Dries R. *et al.* (2021) Giotto: a toolbox for integrative analysis and visualization of spatial expression data. *Genome Biology* 22:78.
- Cable D. M. *et al.* (2022) Robust decomposition of cell type mixtures in spatial transcriptomics. *Nature Biotechnology* 40:517–526.
- Kleshchevnikov V. *et al.* (2022) Cell2location maps fine-grained cell types in spatial transcriptomics. *Nature Biotechnology* 40:661–671.
- Jin S. *et al.* (2021) Inference and analysis of cell-cell communication using CellChat. *Nature Communications* 12:1088.
- Lee D. D., Seung H. S. (1999) Learning the parts of objects by non-negative matrix factorization. *Nature* 401:788–791.
- Lee D. D., Seung H. S. (2001) Algorithms for non-negative matrix factorization. *Advances in Neural Information Processing Systems (NIPS)* 13.

## ➡️ Next

We have now measured RNA in dissociated cells, along time and in space. A cell is more than its RNA: its surface
proteins and its open chromatin carry information the transcriptome misses.
[Module 34 · Multimodal single-cell integration: RNA, protein and chromatin](34_multimodal_integration.ipynb)
combines several measurements of the same cells.